# AutoML Agent Demo Notebook

This notebook demonstrates how to run the AutoML Agent included in this repository with Azure OpenAI models.

## Prerequisites

1. Install dependencies with `pip install -r requirements.txt`.
2. Configure your Azure OpenAI resource and deployments for GPT-4-mini, GPT-5-mini, or o3-mini.
3. Set the environment variables `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_KEY`, and optionally `AZURE_OPENAI_API_VERSION`.

In [ ]:
# Import standard library module for environment variables
import os  # Provides access to environment variables and system settings
# Import configuration objects and model registry from the project
from configs import Configs, AVAILABLE_LLMs  # Gives access to editable configuration values and LLM catalog
# Import the AgentManager orchestrator that coordinates the AutoML agents
from agent_manager import AgentManager  # Main entry point for running the AutoML workflow


In [ ]:
# Retrieve the Azure endpoint from the environment with a descriptive fallback
Configs.AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "https://your-resource.openai.azure.com/")  # Update endpoint configuration
# Retrieve the Azure API key from the environment with a placeholder default
Configs.AZURE_OPENAI_KEY = os.environ.get("AZURE_OPENAI_KEY", "your-azure-openai-key")  # Update API key configuration
# Retrieve the API version while preserving the default if the variable is not set
Configs.AZURE_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", Configs.AZURE_API_VERSION)  # Update API version configuration
# Define deployment names so they can be customized per Azure resource
azure_deployments = {  # Map logical model keys to Azure deployment names
    "azure-gpt-4-mini": os.environ.get("AZURE_GPT4_MINI_DEPLOYMENT", "gpt-4-mini"),  # Default GPT-4-mini deployment name
    "azure-gpt-5-mini": os.environ.get("AZURE_GPT5_MINI_DEPLOYMENT", "gpt-5-mini"),  # Default GPT-5-mini deployment name
    "azure-o3-mini": os.environ.get("AZURE_O3_MINI_DEPLOYMENT", "o3-mini"),  # Default o3-mini deployment name
}  # Close the mapping literal
# Loop through each Azure model to refresh runtime configuration
for model_key, deployment_name in azure_deployments.items():  # Iterate over logical models and deployment names
    AVAILABLE_LLMs[model_key]["endpoint"] = Configs.AZURE_OPENAI_ENDPOINT  # Ensure endpoint is up to date
    AVAILABLE_LLMs[model_key]["api_key"] = Configs.AZURE_OPENAI_KEY  # Ensure API key is up to date
    AVAILABLE_LLMs[model_key]["api_version"] = Configs.AZURE_API_VERSION  # Ensure API version is up to date
    AVAILABLE_LLMs[model_key]["model"] = deployment_name  # Apply user-specified Azure deployment name
# Provide feedback so the user knows which deployments are active
print("Configured Azure deployments:")  # Notify user that configuration succeeded
for model_key, deployment_name in azure_deployments.items():  # Iterate again to display mappings
    print(f"  {model_key} -> {deployment_name}")  # Show logical name to deployment mapping
# Warn the user if placeholder values are still present
if "your-resource.openai.azure.com" in Configs.AZURE_OPENAI_ENDPOINT or "your-azure-openai-key" in Configs.AZURE_OPENAI_KEY:  # Check for placeholder configuration
    print("⚠️ Please replace placeholder Azure credentials with your actual resource details.")  # Display warning when placeholders remain


In [ ]:
# Define a helper function to run the AutoML workflow with extensive comments
def run_automl_task(task_prompt, data_path, llm_choice="azure-gpt-4-mini", **agent_kwargs):  # Expose parameters for flexibility
    # Execute the AutoML Agent with the provided configuration
    manager = AgentManager(  # Instantiate the orchestrator
        task=agent_kwargs.pop("task_name", "automl_project"),  # Supply a task identifier required by AgentManager
        llm=llm_choice,  # Select which Azure model to use
        data_path=data_path,  # Provide the dataset location (local path or URL)
        interactive=agent_kwargs.pop("interactive", False),  # Allow optional interactive refinement
        n_plans=agent_kwargs.pop("n_plans", 3),  # Control how many solution plans to generate
        n_candidates=agent_kwargs.pop("n_candidates", 3),  # Control how many candidate solutions to explore
        n_revise=agent_kwargs.pop("n_revise", 3),  # Control how many revision attempts are allowed
        decomp=agent_kwargs.pop("decomp", True),  # Toggle decomposition of the workflow
        verification=agent_kwargs.pop("verification", True),  # Toggle verification of generated code
        full_pipeline=agent_kwargs.pop("full_pipeline", True),  # Decide whether to execute the entire pipeline
    )  # Finish the AgentManager instantiation
    if agent_kwargs:  # Detect if unsupported arguments were provided
        print(f"Ignored unused parameters: {agent_kwargs}")  # Inform the user about unused arguments
    manager.initiate_chat(prompt=task_prompt)  # Trigger the AutoML workflow with the user prompt
    return manager  # Return the manager instance for inspection after execution


## Example Usage

Adjust the parameters below to match your dataset and Azure deployment preferences.

In [ ]:
# Describe the dataset and task for the AutoML Agent
example_prompt = "Build a classification model to predict customer churn with feature engineering and evaluation reports."  # Sample AutoML instruction
# Point to your dataset location (can be a local file path or URL)
example_data_path = "data/tabular/customer_churn.csv"  # Replace with the path to your dataset
# Choose which Azure model should power the agents (including the new GPT-5-mini option)
example_llm_choice = "azure-gpt-5-mini"  # Select azure-gpt-5-mini to leverage the latest deployment
# Execute the AutoML workflow when you are ready (commented out to avoid accidental execution without credentials)
# run_automl_task(example_prompt, example_data_path, llm_choice=example_llm_choice)  # Uncomment to launch the agent


## Notes

- Ensure you have valid Azure OpenAI credentials before running the workflow.
- The agent will create output artifacts inside the `agent_workspace` directory.
- Switch `example_llm_choice` between `azure-gpt-4-mini`, `azure-gpt-5-mini`, and `azure-o3-mini` to compare model behaviors.